In [6]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch 
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision
from torch.utils.data import DataLoader
import numpy as np
import torch_directml
import torch.nn.functional as F

In [ ]:
device=('cuda' if torch.cuda.is_available() else 'cpu')

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_data=torchvision.datasets.CIFAR10(root='./data',train=True,transform=transform,download=True)
test_data=torchvision.datasets.CIFAR10(root='./data',train=False,transform=transform,download=True)


train_loader=DataLoader(train_data,batch_size=4,shuffle=True)
test_loader=DataLoader(test_data,batch_size=4,shuffle=False)

classess=('plain','card','bird','cat','deer','dog','frog','horse','ship','truck')

class MyConv(nn.Module):
    def __init__(self):
        super(MyConv,self).__init__()
        self.conv1=nn.Conv2d(3,6,5)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(6,16,5)

        self.fc1=nn.Linear(16*5*5,120)
        self.fc2=nn.Linear(120,64)
        self.fc3=nn.Linear(64,10)


    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,16*5*5)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x


model=MyConv().to(device)

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.001)

model.train()
for i in range(10):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        
        output=model(images)
        loss=loss_fn(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f' epoch : {i+1} , loss : {loss.item():.4f}')



Files already downloaded and verified
Files already downloaded and verified
 epoch : 1 , loss : 2.3825
 epoch : 2 , loss : 1.2261
 epoch : 3 , loss : 1.3971
 epoch : 4 , loss : 1.6847
 epoch : 5 , loss : 0.8218
 epoch : 6 , loss : 0.8397
 epoch : 7 , loss : 1.4749
 epoch : 8 , loss : 1.3255
 epoch : 9 , loss : 1.3516
 epoch : 10 , loss : 0.9822


In [14]:
model.eval()
with torch.no_grad():
    total=0
    correct=0
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)

        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print('Accuracy : ' , round(acc,2),'%')
        

Accuracy :  57.42 %


Updated Files

In [ ]:

device=('cuda' if torch.cuda.is_available() else 'cpu')

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


train_data=torchvision.datasets.CIFAR10(root='./data',train=True,transform=transform_train,download=True)
test_data=torchvision.datasets.CIFAR10(root='./data',train=False,transform=transform_test,download=True)


train_loader=DataLoader(train_data,batch_size=64,shuffle=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False)

classess=('plain','card','bird','cat','deer','dog','frog','horse','ship','truck')

class MyConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64*8*8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


model=MyConv().to(device)

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

model.train()
for i in range(30):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        
        output=model(images)

        loss=loss_fn(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f' epoch : {i+1} , loss : {loss.item():.4f}')


model.eval()
with torch.no_grad():
    total=0
    correct=0
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)

        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print('Accuracy : ' , round(acc,2),'%')

Using Resnet18

In [ ]:


from torchvision.models import resnet18

# ----------------------------
# Device
# ----------------------------

print("Using device:", device)

# ----------------------------
# Transforms (Data Augmentation)
# ----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# ----------------------------
# Dataset
# ----------------------------
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

# ----------------------------
# Model (ResNet18)
# ----------------------------
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# ----------------------------
# Loss & Optimizer
# ----------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50
)

# ----------------------------
# Training
# ----------------------------
epochs = 50
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f}")

    # ----------------------------
    # Evaluation
    # ----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pth")
        print(" Model saved")

print(f"\nBest Accuracy Achieved: {best_acc:.2f}%")
